# dim_rls_usuarios — construção

Constrói `lake_gold_fatos.dbo.dim_rls_usuarios` e grava via Spark.

---

## Contexto

Modelo semântico Fabric (SESC-SP) com RLS baseado nesta tabela.
Ambiente: Microsoft Fabric — `lake_gold_fatos` (Warehouse), `lake_prep_siplan` (Lakehouse).

---

## Pipeline de atualização (ordem obrigatória)

1. Dataflow `df_corr_rls` → lê SharePoint → grava delta tables em `lake_prep_siplan`
2. **Este notebook** → constrói `dim_rls_usuarios` → grava em `lake_gold_fatos`
3. Refresh do modelo semântico

---

## Listas SharePoint

Site: `https://sescsp.sharepoint.com/sites/GTDadosSTS`

### rls_grupos — perfis estáveis
| Coluna | Tipo SP | Observação |
|---|---|---|
| Título | texto | nome legível |
| e-mail | texto | UPN — notebook renomeia para `email` |
| perfil_acesso | escolha | `DADOS_GEDES`, `REPRESENTANTE` |
| escopo | escolha | `UNIDADE`, `GERENTE_SEDE`, `ADMIN_CENTRAL` |
| unidade | texto | código UO, `all`, ou pipe-separado (`68\|77`) → notebook explode |
| gerencias | texto | pipe-separado ou `all` |

### rls_overrides — exceções individuais
| Coluna | Tipo SP | Observação |
|---|---|---|
| Título | texto | nome legível |
| e-mail | texto | UPN — notebook renomeia para `email` |
| perfil_acesso | escolha | qualquer perfil exceto GERAL |
| escopo | escolha | `UNIDADE`, `GERENTE_SEDE`, `ADMIN_CENTRAL` |
| unidade | texto | código UO ou `all` — vem como float do Dataflow (`82.0`) → `_clean_unidade()` converte |
| gerencias | texto | pipe-separado ou `all` |
| motivo | texto | obrigatório |
| data_fim | data | opcional — filtrado automaticamente |
| ativo | sim/não | filtrado automaticamente |

**Multi-unidade:** usar múltiplas linhas por email (uma por unidade) — não pipe-separado em `unidade`.

---

## Regras de derivação de dim_funcionarios

### escopo
| Critério | escopo |
|---|---|
| cargo ∋ "GERENTE" AND secao ∋ "SEDE" | `GERENTE_SEDE` |
| secao ∋ "SEDE" (demais cargos) | `ADMIN_CENTRAL` |
| demais | `UNIDADE` |

### perfil_acesso
| Critério | perfil |
|---|---|
| cargo ∋ "GERENTE" | `GERENTE` |
| cargo ∋ "COORD" | `COORDENADOR` |
| cargo ∋ "SUPERV" | `SUPERVISOR` |
| secao ∋ "PROG" AND escopo = UNIDADE | `PROGRAMADOR` |
| secao ∋ "SEDE" AND cargo ∋ "TÉCNIC"/"ESPECIAL" | `ASSISTENTE` |
| fallback | `GERAL` |

---

## Precedência das fontes

1. `rls_grupos` → **substitui** a classificação oficial para os emails presentes
2. `rls_overrides` → **acrescenta** linhas (preserva oficial + adiciona exceções)
3. `dim_funcionarios` → base para todos os demais (`GERAL` = fallback automático)

Convenção: `unidade = "all"` e `gerencias = "all"` = acesso irrestrito naquela dimensão.

---

## Filtros DAX no modelo semântico

Role: `RLS_Padrao` — atribuir todos os Viewers do workspace.
Admins/Members não são filtrados (comportamento padrão do Power BI).

### dim_unidade (campo `uo` — inteiro)
```dax
VAR _email  = LOWER(USERPRINCIPALNAME())
VAR _linhas = FILTER(dim_rls_usuarios, dim_rls_usuarios[email] = _email)
RETURN
    IF(
        COUNTROWS(_linhas) = 0, FALSE(),
        IF(
            COUNTROWS(FILTER(_linhas, dim_rls_usuarios[unidade] = "all")) > 0, TRUE(),
            COUNTROWS(FILTER(_linhas, dim_rls_usuarios[unidade] = (dim_unidade[uo] & ""))) > 0
        )
    )
```
Nota: `uo` é inteiro → `(dim_unidade[uo] & "")` converte para texto antes de comparar.
Nota: `VALUES()` em variável de tabela causa erro em RLS → usar `COUNTROWS(FILTER(...))`.

### dim_gerencia (campo `sigla` — texto)
```dax
VAR _email   = LOWER(USERPRINCIPALNAME())
VAR _linhas  = FILTER(dim_rls_usuarios, dim_rls_usuarios[email] = _email)
VAR _gerlist = CONCATENATEX(_linhas, dim_rls_usuarios[gerencias], "|")
RETURN
    IF(
        COUNTROWS(_linhas) = 0, FALSE(),
        IF(
            CONTAINSSTRING(_gerlist, "all"), TRUE(),
            CONTAINSSTRING("|" & _gerlist & "|", "|" & dim_gerencia[sigla] & "|")
        )
    )
```

---

## Problemas conhecidos (já resolvidos)

- **Graph API no Fabric**: `mssparkutils.credentials.getToken('https://graph.microsoft.com')` falha com 500.
  Solução: Dataflow Gen2 lê SharePoint e grava delta tables; notebook lê delta via `spark.sql`.
- **Float no Dataflow**: campo `unidade` vindo do SharePoint como `"82.0"` → `_clean_unidade()` converte para `"82"`.
- **Pipe-separado em unidade**: `rls_grupos` com `"68|77"` → notebook explode em duas linhas.
- **e-mail com hífen**: nome interno do campo no SharePoint é `"e-mail"` → renomeado para `"email"` no notebook.

In [ ]:
import pandas as pd
from datetime import date

# ── Parâmetros ────────────────────────────────────────────────────────────────
# Ajustar conforme schema real de dim_funcionarios
COL_EMAIL     = 'email'
COL_CARGO     = 'cargo'
COL_SECAO     = 'secao'
COL_UNIDADE   = 'uo'   # coluna com código UO — ajustar se o nome for diferente
COL_GERENCIAS = None        # coluna com gerências pipe-separadas, ou None se não existir

In [ ]:
# ── Leitura das fontes ────────────────────────────────────────────────────────
cols_func = [COL_EMAIL, COL_CARGO, COL_SECAO, COL_UNIDADE]
if COL_GERENCIAS:
    cols_func.append(COL_GERENCIAS)

dim_func_df      = spark.sql(f"SELECT {', '.join(cols_func)} FROM lake_gold_fatos.dbo.dim_funcionarios").toPandas()
grupos_raw_df    = spark.sql("SELECT * FROM lake_prep_siplan.dbo.rls_grupos").toPandas()
overrides_raw_df = spark.sql("SELECT * FROM lake_prep_siplan.dbo.rls_overrides").toPandas()

print(f'dim_funcionarios : {len(dim_func_df):,}')
print(f'rls_grupos       : {len(grupos_raw_df):,}')
print(f'rls_overrides    : {len(overrides_raw_df):,}')
print()
print('Colunas grupos   :', grupos_raw_df.columns.tolist())
print('Colunas overrides:', overrides_raw_df.columns.tolist())

In [ ]:
# ── Classificação de dim_funcionarios ─────────────────────────────────────────
def _escopo(row):
    cargo = str(row[COL_CARGO] or '').upper()
    secao = str(row[COL_SECAO] or '').upper()
    if 'GERENTE' in cargo and 'SEDE' in secao:
        return 'GERENTE_SEDE'
    if 'SEDE' in secao:
        return 'ADMIN_CENTRAL'
    return 'UNIDADE'

def _perfil(row):
    cargo  = str(row[COL_CARGO] or '').upper()
    secao  = str(row[COL_SECAO] or '').upper()
    escopo = row['escopo']
    if 'GERENTE' in cargo:                                                     return 'GERENTE'
    if 'COORD'   in cargo:                                                     return 'COORDENADOR'
    if 'SUPERV'  in cargo:                                                     return 'SUPERVISOR'
    if 'PROG'    in secao and escopo == 'UNIDADE':                             return 'PROGRAMADOR'
    if 'SEDE'    in secao and any(k in cargo for k in ('TÉCNIC', 'ESPECIAL')): return 'ASSISTENTE'
    return 'GERAL'

oficial_df = dim_func_df.copy()
oficial_df['escopo']        = oficial_df.apply(_escopo, axis=1)
oficial_df['perfil_acesso'] = oficial_df.apply(_perfil, axis=1)
oficial_df['unidade'] = oficial_df.apply(
    lambda r: str(r[COL_UNIDADE]) if r['escopo'] == 'UNIDADE' and pd.notna(r[COL_UNIDADE]) else 'all',
    axis=1,
)
oficial_df['gerencias'] = oficial_df.apply(
    lambda r: (str(r[COL_GERENCIAS]) if COL_GERENCIAS and pd.notna(r.get(COL_GERENCIAS)) else 'all')
              if r['escopo'] == 'GERENTE_SEDE' else 'all',
    axis=1,
)
oficial_df['fonte'] = 'OFICIAL'
oficial_df = oficial_df.rename(columns={COL_EMAIL: 'email'})
oficial_df = oficial_df[['email', 'escopo', 'perfil_acesso', 'unidade', 'gerencias', 'fonte']]

print(oficial_df['perfil_acesso'].value_counts().to_string())

In [ ]:
def _clean_unidade(val):
    s = str(val).strip() if pd.notna(val) else 'all'
    if s in ('', 'nan'):
        return 'all'
    try:
        return str(int(float(s)))  # '82.0' → '82'
    except ValueError:
        return s  # 'all' ou pipe-separado: mantém para explodir depois

# ── Preparar rls_grupos ───────────────────────────────────────────────────────
grupos_df = grupos_raw_df.rename(columns={'e-mail': 'email'})
grupos_df = grupos_df[['email', 'perfil_acesso', 'escopo', 'unidade', 'gerencias']].copy()
grupos_df['unidade'] = grupos_df['unidade'].apply(_clean_unidade)
# unidade pipe-separada ('68|77') → múltiplas linhas ('68', '77')
grupos_df = grupos_df.assign(unidade=grupos_df['unidade'].str.split('|')).explode('unidade')
grupos_df['unidade'] = grupos_df['unidade'].str.strip()
grupos_df['fonte'] = 'GRUPO'

# ── Preparar rls_overrides (filtrar ativos e dentro da validade) ──────────────
ov = overrides_raw_df.rename(columns={'e-mail': 'email'})
ov = ov[['email', 'perfil_acesso', 'escopo', 'unidade', 'gerencias', 'data_fim', 'ativo']].copy()
ov['unidade'] = ov['unidade'].apply(_clean_unidade)
hoje = pd.Timestamp(date.today())
ov['data_fim'] = pd.to_datetime(ov['data_fim'], errors='coerce')
ov['ativo']    = ov['ativo'].astype(str).str.upper().isin(['TRUE', '1', 'SIM', 'YES'])
overrides_df = ov[
    ov['ativo'] & (ov['data_fim'].isna() | (ov['data_fim'] >= hoje))
][['email', 'escopo', 'perfil_acesso', 'unidade', 'gerencias']].copy()
overrides_df['fonte'] = 'OVERRIDE'

print(f'grupos_df    : {len(grupos_df)}')
print(f'overrides_df : {len(overrides_df)}')

In [ ]:
# ── Combinar e gravar ─────────────────────────────────────────────────────────
emails_grupos = set(grupos_df['email'].str.strip().str.lower())
base_df = oficial_df[~oficial_df['email'].str.strip().str.lower().isin(emails_grupos)].copy()

dim_rls_df = pd.concat([
    base_df,
    grupos_df[['email', 'escopo', 'perfil_acesso', 'unidade', 'gerencias', 'fonte']],
    overrides_df,
], ignore_index=True)

dim_rls_df['email'] = dim_rls_df['email'].str.strip().str.lower()
dim_rls_df = dim_rls_df.drop_duplicates()

# Garante que colunas string não tenham float NaN (causa falha no Arrow/Spark)
str_cols = ['email', 'escopo', 'perfil_acesso', 'unidade', 'gerencias', 'fonte']
dim_rls_df[str_cols] = dim_rls_df[str_cols].fillna('').astype(str)

print(f'dim_rls_usuarios: {len(dim_rls_df):,} registros')
print()
print(dim_rls_df['perfil_acesso'].value_counts().to_string())
print()
print(dim_rls_df['escopo'].value_counts().to_string())
print()
print(dim_rls_df['fonte'].value_counts().to_string())

spark.createDataFrame(dim_rls_df) \
     .write.mode('overwrite') \
     .option('overwriteSchema', 'true') \
     .saveAsTable('lake_gold_fatos.dbo.dim_rls_usuarios')

print()
print('Gravado: lake_gold_fatos.dbo.dim_rls_usuarios')